In [ ]:
import sys
sys.path.append('../src')  # if running from notebooks/

from data_loader import load_raw_tweets, list_brands

df = load_raw_tweets('../data/raw/tweets_100k.csv')
print("Shape:", df.shape)
print(df.columns.tolist())
print(df.info())
print(df.head())

In [ ]:
# Count inbound vs outbound
print(df['inbound'].value_counts())

# List top brands (by number of replies)
brand_counts = list_brands(df, min_tweets=1000)
print(brand_counts.head(20))

In [ ]:
from preprocess import extract_pairs

# Choose a brand ID from the top list
# e.g., brand_id = brand_counts.index[0]   # top brand
brand_id = "AmazonHelp" # replace with actual

pairs = extract_pairs(df, brand_id)
print(f"Number of one‑turn pairs: {len(pairs)}")
pairs.head()

In [ ]:
# Assuming you already loaded df
brand_counts = df[df['inbound'] == False]['author_id'].value_counts()
top_brand_id = brand_counts.index[0]   # e.g., 'AmazonHelp'
print("Top brand ID:", top_brand_id)

In [ ]:
import sys
from pathlib import Path

# Add the project root to sys.path so we can import from src/
project_root = Path.cwd().parent   # If you're in notebooks/ ; if you're in project root, use Path.cwd()
sys.path.append(str(project_root))

# Now you can import
from src.preprocess import extract_pairs

In [ ]:
import pandas as pd
from src.preprocess import extract_pairs

# Load your 100k sample
df = pd.read_csv('../data/raw/tweets_100k.csv')

# Quick sanity check: do any IDs match?
brand_replies_test = df[(df['inbound'] == False) & (df['author_id'] == 'AmazonHelp')]
customer_ids = brand_replies_test['in_response_to_tweet_id'].dropna().astype(str).str.strip().unique()
customer_tweets_test = df[df['inbound'] == True].copy()
customer_tweets_test['tweet_id'] = customer_tweets_test['tweet_id'].astype(str).str.strip()
matching = customer_tweets_test[customer_tweets_test['tweet_id'].isin(customer_ids)]
print(f"Number of matching customer tweets: {len(matching)}")

# Now call the function
pairs = extract_pairs(df, 'AmazonHelp')
print(f"Number of one-turn pairs: {len(pairs)}")
pairs.head()

In [ ]:
df = pd.read_csv('../data/raw/amazon_tweets_100k.csv', dtype={'tweet_id': str, 'in_response_to_tweet_id': str})

In [ ]:
import pandas as pd

# Load the new dataset with string IDs
df = pd.read_csv('../data/raw/amazon_tweets_100k.csv', dtype={'tweet_id': str, 'in_response_to_tweet_id': str})

# Quick sanity check
brand_replies = df[(df['inbound'] == False) & (df['author_id'] == 'AmazonHelp')]
customer_ids = brand_replies['in_response_to_tweet_id'].dropna().unique()
customer_tweets = df[df['inbound'] == True]
matching = customer_tweets[customer_tweets['tweet_id'].isin(customer_ids)]
print(f"Number of matching customer tweets: {len(matching)}")

# Now extract pairs using the updated function from earlier
from src.preprocess import extract_pairs
pairs = extract_pairs(df, 'AmazonHelp')
print(f"Number of one-turn pairs: {len(pairs)}")
print(pairs.head())

In [ ]:
import sys
from pathlib import Path

# Add the project root to sys.path so we can import from src/
project_root = Path.cwd().parent   # If you're in notebooks/ ; if you're in project root, use Path.cwd()
sys.path.append(str(project_root))

# Now you can import
from src.preprocess import clean_text, extract_pairs

In [29]:
with open('../src/preprocess.py', 'r') as f:
    content = f.read()
print(content)

import re
import pandas as pd

def extract_pairs(df, brand_id):
    """
    Extract one-turn pairs (customer message + brand reply) for a given brand.
    """
    df = df.copy()
    df['tweet_id'] = df['tweet_id'].astype(str).str.strip()
    df['in_response_to_tweet_id'] = df['in_response_to_tweet_id'].astype(str).str.strip()

    brand_replies = df[(df['inbound'] == False) & (df['author_id'] == brand_id)].copy()
    brand_replies = brand_replies[brand_replies['in_response_to_tweet_id'] != 'nan']
    brand_replies = brand_replies[brand_replies['in_response_to_tweet_id'] != '']

    customer_tweet_ids = brand_replies['in_response_to_tweet_id'].unique()
    customer_tweets = df[(df['inbound'] == True) & (df['tweet_id'].isin(customer_tweet_ids))].copy()

    pairs = customer_tweets.merge(
        brand_replies[['in_response_to_tweet_id', 'text', 'created_at']],
        left_on='tweet_id',
        right_on='in_response_to_tweet_id',
        how='inner',
        suffixes=('_cust', '_brand')

In [30]:
print(repr(content))

'import re\nimport pandas as pd\n\ndef extract_pairs(df, brand_id):\n    """\n    Extract one-turn pairs (customer message + brand reply) for a given brand.\n    """\n    df = df.copy()\n    df[\'tweet_id\'] = df[\'tweet_id\'].astype(str).str.strip()\n    df[\'in_response_to_tweet_id\'] = df[\'in_response_to_tweet_id\'].astype(str).str.strip()\n\n    brand_replies = df[(df[\'inbound\'] == False) & (df[\'author_id\'] == brand_id)].copy()\n    brand_replies = brand_replies[brand_replies[\'in_response_to_tweet_id\'] != \'nan\']\n    brand_replies = brand_replies[brand_replies[\'in_response_to_tweet_id\'] != \'\']\n\n    customer_tweet_ids = brand_replies[\'in_response_to_tweet_id\'].unique()\n    customer_tweets = df[(df[\'inbound\'] == True) & (df[\'tweet_id\'].isin(customer_tweet_ids))].copy()\n\n    pairs = customer_tweets.merge(\n        brand_replies[[\'in_response_to_tweet_id\', \'text\', \'created_at\']],\n        left_on=\'tweet_id\',\n        right_on=\'in_response_to_tweet_id\',

In [31]:
%reset -f


In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.preprocess import extract_pairs, clean_text
print("Import successful")

Import successful


In [3]:
import sys
from pathlib import Path
import pandas as pd

# Add project root to path (so we can import from src/)
project_root = Path.cwd().parent   # if your notebook is in 'notebooks/'
sys.path.append(str(project_root))

from src.preprocess import extract_pairs, clean_text

# 2. Load raw data (adjust path if needed)
df = pd.read_csv('../data/raw/amazon_tweets_100k.csv', dtype={'tweet_id': str, 'in_response_to_tweet_id': str})

# 3. Extract one-turn pairs
pairs = extract_pairs(df, 'AmazonHelp')
print(f"Total pairs: {len(pairs)}")

# 4. Clean the text
pairs['customer_msg_clean'] = pairs['customer_msg'].apply(clean_text)
pairs['brand_reply_clean'] = pairs['brand_reply'].apply(clean_text)
print("Cleaning complete.")

Total pairs: 6893
Cleaning complete.


In [4]:
pairs.to_csv('../data/processed/pairs_cleaned_full.csv', index=False)
print("Saved full cleaned pairs.")

Saved full cleaned pairs.
